
# Classification binaire des compétences IA / non-IA — Machine Learning

Objectif : entraîner un classifieur binaire sur le même dataset, avec le même split stratifié, la même seed et les mêmes métriques que le notebook TextCNN.

Règles appliquées ici :

- classe `0` = non-IA ;
- classe `1` = IA ;
- split stratifié 70 % / 15 % / 15 % ;
- seuil choisi uniquement sur validation ;
- évaluation finale effectuée une seule fois sur test.



```bash
pip install pandas numpy scikit-learn matplotlib openpyxl joblib torch
```


In [25]:

from __future__ import annotations

import json
import sys
import time
import unicodedata
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

ROOT = None
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / '.git').exists() and (candidate / 'data' / 'raw').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Impossible de localiser la racine du dépôt.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.ia_non_ia_shared import (  # noqa: E402
    SEED,
    audit_dataset,
    build_threshold_table,
    build_tfidf_logistic_regression_pipeline,
    choose_threshold_from_validation,
    enrich_with_ids,
    evaluate_with_threshold,
    find_repo_root,
    load_or_create_splits,
    model_size_bytes,
    package_versions,
    plot_class_distribution,
    plot_confusion_matrix,
    plot_length_distribution,
    plot_probability_distribution,
    plot_roc_pr_calibration,
    plot_threshold_metrics,
    read_dataset,
    report_to_flat_row,
    save_dataframe,
    save_json,
    split_frames,
    summarize_split_sizes,
    threshold_grid,
    extract_logistic_top_features,
)

ROOT = find_repo_root(ROOT)
DATASET_PATH = ROOT / 'data' / 'raw' / 'dataset_competences_IA_annotees.xlsx'
ARTIFACT_DIR = ROOT / 'artifacts' / 'classification_ia_non_ia_ml'
COMMON_SPLIT_PATH = ROOT / 'artifacts' / 'classification_ia_non_ia_common' / 'splits.csv'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
COMMON_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = None
LABEL_COLUMN = None
C_VALUE = 1.0
THRESHOLD_GRID = threshold_grid(0.05, 0.95, 0.01)


In [ ]:

def normalize_name(value: str) -> str:
    normalized = unicodedata.normalize('NFKD', str(value))
    normalized = ''.join(ch for ch in normalized if not unicodedata.combining(ch))
    return normalized.lower().strip()


def detect_columns(frame: pd.DataFrame) -> tuple[str, str]:
    normalized = {normalize_name(column): column for column in frame.columns}
    text_candidates = ['competence', 'compétence', 'texte', 'text']
    label_candidates = ['ia', 'label', 'cible', 'target', 'classe']
    text_column = next((normalized[name] for name in text_candidates if name in normalized), None)
    label_column = next((normalized[name] for name in label_candidates if name in normalized), None)
    if text_column is None or label_column is None:
        raise ValueError(f'Colonnes attendues introuvables. Colonnes disponibles: {list(frame.columns)}')
    return text_column, label_column


def to_binary_frame(frame: pd.DataFrame, text_column: str, label_column: str) -> pd.DataFrame:
    output = frame.copy()
    output['text'] = output[text_column].astype(str)
    output['is_ai'] = output[label_column].astype(int)
    return output[['source_index', 'record_id', text_column, label_column, 'split', 'text', 'is_ai']].copy()


def summarize_threshold_table(table: pd.DataFrame) -> pd.DataFrame:
    cols = ['threshold', 'precision_ia', 'recall_ia', 'f1_ia', 'f1_macro']
    return table[cols].copy().sort_values('threshold').reset_index(drop=True)


def run_cross_validation(train_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_rows: list[dict[str, float]] = []
    feature_texts = train_frame['text'].tolist()
    feature_labels = train_frame['is_ai'].to_numpy(dtype=int)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(feature_texts, feature_labels), start=1):
        fold_train = train_frame.iloc[train_idx]
        fold_valid = train_frame.iloc[valid_idx]

        pipeline = build_tfidf_logistic_regression_pipeline(C=C_VALUE, seed=SEED)
        start_fit = time.perf_counter()
        pipeline.fit(fold_train['text'], fold_train['is_ai'])
        training_time = time.perf_counter() - start_fit

        start_inference = time.perf_counter()
        valid_scores = pipeline.predict_proba(fold_valid['text'])[:, 1]
        inference_time = time.perf_counter() - start_inference
        threshold_table = build_threshold_table(fold_valid['is_ai'], valid_scores, thresholds=THRESHOLD_GRID)
        threshold, best_row = choose_threshold_from_validation(threshold_table)
        metrics, _ = evaluate_with_threshold(
            fold_valid['is_ai'],
            valid_scores,
            threshold=threshold,
            model_name='tfidf_logistic_regression_cv',
            inference_time_seconds=inference_time,
            model_size_bytes=None,
        )
        metrics['training_time_seconds'] = training_time
        metrics['fold'] = fold
        metrics['threshold'] = threshold
        fold_rows.append(metrics)

    fold_metrics = pd.DataFrame(fold_rows)

    summary_rows = []
    metric_cols = [
        'accuracy', 'balanced_accuracy', 'precision_ia', 'recall_ia', 'f1_ia',
        'precision_non_ia', 'recall_non_ia', 'f1_non_ia', 'precision_macro', 'recall_macro',
        'f1_macro', 'f1_weighted', 'roc_auc', 'pr_auc', 'mcc', 'cohen_kappa', 'log_loss',
        'brier_score', 'specificity', 'negative_predictive_value', 'tp', 'fp', 'tn', 'fn',
        'training_time_seconds', 'inference_time_seconds', 'latency_ms_per_sample', 'threshold',
    ]
    for metric in metric_cols:
        values = fold_metrics[metric].astype(float)
        summary_rows.append({
            'metric': metric,
            'fold_1': float(values.iloc[0]),
            'fold_2': float(values.iloc[1]),
            'fold_3': float(values.iloc[2]),
            'fold_4': float(values.iloc[3]),
            'fold_5': float(values.iloc[4]),
            'mean': float(values.mean()),
            'std': float(values.std(ddof=0)),
            'min': float(values.min()),
            'max': float(values.max()),
        })
    summary = pd.DataFrame(summary_rows)
    return fold_metrics, summary


def select_best_threshold_and_metrics(valid_frame: pd.DataFrame, valid_scores: np.ndarray) -> tuple[float, pd.DataFrame, dict[str, float], object]:
    threshold_table = build_threshold_table(valid_frame['is_ai'], valid_scores, thresholds=THRESHOLD_GRID)
    threshold, best_row = choose_threshold_from_validation(threshold_table)
    metrics, report = evaluate_with_threshold(
        valid_frame['is_ai'],
        valid_scores,
        threshold=threshold,
        model_name='tfidf_logistic_regression',
        inference_time_seconds=None,
        model_size_bytes=None,
    )
    return threshold, threshold_table, metrics, report


import numpy as np
import pandas as pd


def error_analysis(
    frame: pd.DataFrame,
    scores: np.ndarray,
    threshold: float,
) -> pd.DataFrame:
    frame = frame.copy()
    scores = np.asarray(scores, dtype=float).reshape(-1)

    if len(frame) != len(scores):
        raise ValueError(
            f"Nombre de lignes différent du nombre de scores : "
            f"{len(frame)} lignes, {len(scores)} scores"
        )

    predictions = (scores >= float(threshold)).astype(int)

    # On aligne les tableaux NumPy sur les vrais index pandas.
    score_series = pd.Series(
        scores,
        index=frame.index,
        name="probability_ia",
    )

    prediction_series = pd.Series(
        predictions,
        index=frame.index,
        name="predicted_class",
    )

    true_labels = frame["is_ai"].astype(int)

    error_mask = prediction_series.ne(true_labels)
    errors = frame.loc[error_mask].copy()

    output_columns = [
        "source_index",
        "record_id",
        "split",
        "text",
        "true_label",
        "predicted_label",
        "probability_ia",
        "threshold",
        "error_type",
    ]

    if errors.empty:
        return pd.DataFrame(columns=output_columns)

    errors["true_label"] = (
        errors["is_ai"]
        .astype(int)
        .map({0: "non-IA", 1: "IA"})
    )

    errors["predicted_label"] = (
        prediction_series
        .loc[errors.index]
        .map({0: "non-IA", 1: "IA"})
        .to_numpy()
    )

    errors["probability_ia"] = (
        score_series
        .loc[errors.index]
        .to_numpy()
    )

    errors["threshold"] = float(threshold)

    errors["error_type"] = np.where(
        errors["is_ai"].astype(int).eq(1),
        "faux négatif",
        "faux positif",
    )

    # Ajout défensif des colonnes facultatives.
    if "source_index" not in errors.columns:
        errors["source_index"] = errors.index

    if "record_id" not in errors.columns:
        errors["record_id"] = errors["source_index"].astype(str)

    if "split" not in errors.columns:
        errors["split"] = "test"

    if "text" not in errors.columns:
        raise KeyError(
            "La colonne 'text' est absente de df_test. "
            f"Colonnes disponibles : {list(frame.columns)}"
        )

    return (
        errors[output_columns]
        .sort_values("probability_ia", ascending=False)
        .reset_index(drop=True)
    )
def predict_competence_ml(text: str, *, model_dir: Path | None = None) -> dict[str, object]:
    model_dir = model_dir or ARTIFACT_DIR
    pipeline = joblib.load(model_dir / 'tfidf_logistic_regression.joblib')
    threshold_payload = json.loads((model_dir / 'threshold.json').read_text(encoding='utf-8'))
    threshold = float(threshold_payload['threshold'])
    probability = float(pipeline.predict_proba([str(text)])[:, 1][0])
    prediction = 'IA' if probability >= threshold else 'non-IA'
    return {
        'competence': str(text),
        'probability_ia': probability,
        'threshold': threshold,
        'prediction': prediction,
    }


In [27]:

# Chargement et audit du dataset
raw = read_dataset(DATASET_PATH)
text_column, label_column = detect_columns(raw)
TEXT_COLUMN, LABEL_COLUMN = text_column, label_column

audit = audit_dataset(raw, text_column=text_column, label_column=label_column)
print(f"Feuille source: Dataset")
print(f"Colonnes détectées: texte='{text_column}', cible='{label_column}'")
print(f"Nombre de lignes: {audit['n_rows']}")
print(f"Distribution des classes: {audit['label_distribution']}")
print(f"Valeurs manquantes: {audit['missing_by_column']}")
print(f"Doublons exacts: {audit['exact_duplicates']}")
print(f"Contradictions d'étiquettes: {audit['contradictory_labels_by_text']}")
print(f"Textes très courts: {audit['short_text_counts']}")

audit_df = pd.DataFrame({
    'métrique': [
        'Nombre de lignes', 'Nombre de colonnes', 'Distribution classe 0', 'Distribution classe 1',
        'Doublons exacts', "Contradictions d'étiquettes", 'Textes < 5 caractères', 'Textes < 3 mots'
    ],
    'valeur': [
        audit['n_rows'], audit['n_columns'], audit['label_distribution'].get(0, 0), audit['label_distribution'].get(1, 0),
        audit['exact_duplicates'], audit['contradictory_labels_by_text'], audit['short_text_counts']['len_chars_lt_5'], audit['short_text_counts']['len_words_lt_3']
    ]
})
display(audit_df)

display(pd.DataFrame({'colonne': list(audit['missing_by_column'].keys()), 'valeurs_manquantes': list(audit['missing_by_column'].values())}))
display(pd.DataFrame({'label': ['non-IA', 'IA'], 'count': [audit['label_distribution'].get(0, 0), audit['label_distribution'].get(1, 0)], 'ratio': [audit['label_distribution_ratio'].get('0', 0), audit['label_distribution_ratio'].get('1', 0)]}))

fig, ax = plt.subplots(figsize=(5, 4))
counts = pd.Series({0: audit['label_distribution'].get(0, 0), 1: audit['label_distribution'].get(1, 0)})
counts.index = ['non-IA', 'IA']
counts.plot(kind='bar', ax=ax, color=['#4c78a8', '#f58518'])
ax.set_title('Distribution des classes')
ax.set_ylabel("Nombre d'exemples")
ax.set_xlabel('Classe')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
text_lengths = raw[text_column].astype(str).str.strip().str.len()
ax.hist(text_lengths, bins=30, color='#4c78a8', alpha=0.85)
ax.set_title('Distribution des longueurs des compétences')
ax.set_xlabel('Nombre de caractères')
ax.set_ylabel('Fréquence')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
word_lengths = raw[text_column].astype(str).str.strip().str.split().str.len()
ax.hist(word_lengths, bins=20, color='#72b7b2', alpha=0.85)
ax.set_title('Distribution des longueurs en mots')
ax.set_xlabel('Nombre de mots')
ax.set_ylabel('Fréquence')
plt.tight_layout()
plt.show()


Feuille source: Dataset
Colonnes détectées: texte='compétence', cible='IA'
Nombre de lignes: 1800
Distribution des classes: {0: 1200, 1: 600}
Valeurs manquantes: {'compétence': 0, 'IA': 0, 'catégorie IA': 1200, 'compétence IA associée': 600}
Doublons exacts: 0
Contradictions d'étiquettes: 0
Textes très courts: {'len_chars_lt_5': 9, 'len_words_lt_3': 73, 'len_chars_le_10': 53, 'len_words_le_3': 213}


,métrique,valeur
0,Nombre de lignes,1800
1,Nombre de colonnes,4
2,Distribution classe 0,1200
3,Distribution classe 1,600
4,Doublons exacts,0
5,Contradictions d'étiquettes,0
6,Textes < 5 caractères,9
7,Textes < 3 mots,73


,colonne,valeurs_manquantes
0,compétence,0
1,IA,0
2,catégorie IA,1200
3,compétence IA associée,600


,label,count,ratio
0,non-IA,1200,0.666667
1,IA,600,0.333333


/tmp/ipykernel_64762/2255003267.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_64762/2255003267.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_64762/2255003267.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:

# Création du split stratifié 70/15/15 avec conservation des identifiants stables
base = enrich_with_ids(raw)
base = load_or_create_splits(base, label_column=label_column, seed=SEED, split_path=COMMON_SPLIT_PATH)
frames = split_frames(base)

df_train = to_binary_frame(frames['train'], text_column=text_column, label_column=label_column)
df_valid = to_binary_frame(frames['validation'], text_column=text_column, label_column=label_column)
df_test = to_binary_frame(frames['test'], text_column=text_column, label_column=label_column)

print(summarize_split_sizes(base, label_column))
display(pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'n_samples': [len(df_train), len(df_valid), len(df_test)],
    'n_ia': [int(df_train['is_ai'].sum()), int(df_valid['is_ai'].sum()), int(df_test['is_ai'].sum())],
    'n_non_ia': [int((df_train['is_ai'] == 0).sum()), int((df_valid['is_ai'] == 0).sum()), int((df_test['is_ai'] == 0).sum())],
}))

assert set(df_train['is_ai'].unique()) <= {0, 1}
assert set(df_valid['is_ai'].unique()) <= {0, 1}
assert set(df_test['is_ai'].unique()) <= {0, 1}
assert df_train['is_ai'].nunique() == 2 and df_valid['is_ai'].nunique() == 2 and df_test['is_ai'].nunique() == 2


{'sizes': {'train': 1260, 'validation': 270, 'test': 270}, 'by_label': {'train': {0: 840, 1: 420}, 'validation': {0: 180, 1: 90}, 'test': {0: 180, 1: 90}}}


,split,n_samples,n_ia,n_non_ia
0,train,1260,420,840
1,validation,270,90,180
2,test,270,90,180


In [29]:

# Validation croisée stratifiée à 5 folds sur le train uniquement
cv_metrics, cv_summary = run_cross_validation(df_train)
cv_metrics_path = ARTIFACT_DIR / 'cross_validation_metrics.csv'
cv_metrics.to_csv(cv_metrics_path, index=False, encoding='utf-8')
cv_summary_path = ARTIFACT_DIR / 'cross_validation_metrics_summary.csv'
cv_summary.to_csv(cv_summary_path, index=False, encoding='utf-8')

display(cv_metrics[['fold', 'accuracy', 'balanced_accuracy', 'precision_ia', 'recall_ia', 'f1_ia', 'f1_macro', 'roc_auc', 'pr_auc', 'training_time_seconds', 'latency_ms_per_sample', 'threshold']])
display(cv_summary)


,fold,accuracy,balanced_accuracy,precision_ia,recall_ia,f1_ia,f1_macro,roc_auc,pr_auc,training_time_seconds,latency_ms_per_sample,threshold
0,1,0.928571,0.934524,0.851064,0.952381,0.898876,0.921831,0.977820,0.958246,0.049741,0.037687,0.44
1,2,0.928571,0.904762,0.945946,0.833333,0.886076,0.917026,0.977749,0.960441,0.047036,0.035756,0.54
2,3,0.944444,0.937500,0.916667,0.916667,0.916667,0.937500,0.984765,0.967200,0.048075,0.036863,0.53
3,4,0.940476,0.928571,0.925926,0.892857,0.909091,0.932422,0.984269,0.972742,0.046571,0.036259,0.50
4,5,0.900794,0.892857,0.839080,0.869048,0.853801,0.889363,0.942106,0.909519,0.148593,0.037298,0.49


,metric,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std,min,max
0,accuracy,0.928571,0.928571,0.944444,0.940476,0.900794,0.928571,0.015266,0.900794,0.944444
1,balanced_accuracy,0.934524,0.904762,0.937500,0.928571,0.892857,0.919643,0.017658,0.892857,0.937500
2,precision_ia,0.851064,0.945946,0.916667,0.925926,0.839080,0.895737,0.042605,0.839080,0.945946
3,recall_ia,0.952381,0.833333,0.916667,0.892857,0.869048,0.892857,0.040546,0.833333,0.952381
4,f1_ia,0.898876,0.886076,0.916667,0.909091,0.853801,0.892902,0.022082,0.853801,0.916667
5,precision_non_ia,0.974684,0.921348,0.958333,0.947368,0.933333,0.947013,0.018653,0.921348,0.974684
6,recall_non_ia,0.916667,0.976190,0.958333,0.964286,0.916667,0.946429,0.024972,0.916667,0.976190
7,f1_non_ia,0.944785,0.947977,0.958333,0.955752,0.924925,0.946355,0.011799,0.924925,0.958333
8,precision_macro,0.912874,0.933647,0.937500,0.936647,0.886207,0.921375,0.019763,0.886207,0.937500
9,recall_macro,0.934524,0.904762,0.937500,0.928571,0.892857,0.919643,0.017658,0.892857,0.937500


In [30]:

# Entraînement final sur le train, choix du seuil sur validation, évaluation finale sur test
pipeline = build_tfidf_logistic_regression_pipeline(C=C_VALUE, seed=SEED)
start_train = time.perf_counter()
pipeline.fit(df_train['text'], df_train['is_ai'])
training_time_seconds = time.perf_counter() - start_train

start_valid_inference = time.perf_counter()
valid_scores = pipeline.predict_proba(df_valid['text'])[:, 1]
valid_inference_time = time.perf_counter() - start_valid_inference

threshold_table_validation = build_threshold_table(df_valid['is_ai'], valid_scores, thresholds=THRESHOLD_GRID)
threshold, best_threshold_row = choose_threshold_from_validation(threshold_table_validation)
validation_metrics, validation_report = evaluate_with_threshold(
    df_valid['is_ai'],
    valid_scores,
    threshold=threshold,
    model_name='tfidf_logistic_regression',
    inference_time_seconds=valid_inference_time,
    model_size_bytes=None,
)
validation_metrics['training_time_seconds'] = training_time_seconds

start_test_inference = time.perf_counter()
test_scores = pipeline.predict_proba(df_test['text'])[:, 1]
test_inference_time = time.perf_counter() - start_test_inference
test_metrics, test_report = evaluate_with_threshold(
    df_test['is_ai'],
    test_scores,
    threshold=threshold,
    model_name='tfidf_logistic_regression',
    inference_time_seconds=test_inference_time,
    model_size_bytes=None,
)
test_metrics['training_time_seconds'] = training_time_seconds

model_path = ARTIFACT_DIR / 'tfidf_logistic_regression.joblib'
joblib.dump(pipeline, model_path)
model_size_bytes_value = model_size_bytes(model_path)
validation_metrics['model_size_bytes'] = model_size_bytes_value
test_metrics['model_size_bytes'] = model_size_bytes_value
validation_metrics['model_size_mb'] = model_size_bytes_value / (1024 * 1024)
test_metrics['model_size_mb'] = model_size_bytes_value / (1024 * 1024)

threshold_payload = {
    'model_name': 'tfidf_logistic_regression',
    'threshold': float(threshold),
    'selection_rule': [
        'meilleur F1 macro',
        'meilleur F1 IA',
        'meilleur rappel IA',
        "puis seuil le plus proche de 0.5 en cas d'égalité"
    ],
    'threshold_grid': [float(x) for x in THRESHOLD_GRID.tolist()],
    'best_threshold_row': best_threshold_row.to_dict(),
    'validation_threshold_table_rows': int(len(threshold_table_validation)),
    'date': pd.Timestamp.utcnow().isoformat(),
}
save_json(ARTIFACT_DIR / 'threshold.json', threshold_payload)

save_json(ARTIFACT_DIR / 'metrics_validation.json', validation_metrics)
save_json(ARTIFACT_DIR / 'metrics_test.json', test_metrics)

positive_features, negative_features = extract_logistic_top_features(pipeline, top_n=50)
positive_features.to_csv(ARTIFACT_DIR / 'top_features_ia.csv', index=False, encoding='utf-8')
negative_features.to_csv(ARTIFACT_DIR / 'top_features_non_ia.csv', index=False, encoding='utf-8')

errors_test = error_analysis(df_test, test_scores, threshold)
errors_test.to_csv(ARTIFACT_DIR / 'error_analysis.csv', index=False, encoding='utf-8')

metadata = {
    'model_type': 'TF-IDF(word+char)+LogisticRegression',
    'pretrained': False,
    'seed': SEED,
    'date_entrainement': pd.Timestamp.utcnow().isoformat(),
    'colonnes_utilisees': {'texte': text_column, 'cible': label_column},
    'nombre_exemples_par_split': {'train': len(df_train), 'validation': len(df_valid), 'test': len(df_test)},
    'hyperparametres': {
        'word_tfidf': {
            'analyzer': 'word', 'ngram_range': [1, 2], 'min_df': 2, 'max_df': 0.98,
            'max_features': 50000, 'sublinear_tf': True, 'strip_accents': 'unicode'
        },
        'char_tfidf': {
            'analyzer': 'char_wb', 'ngram_range': [3, 5], 'min_df': 2,
            'max_features': 50000, 'sublinear_tf': True, 'strip_accents': 'unicode'
        },
        'logistic_regression': {
            'class_weight': 'balanced', 'max_iter': 3000, 'random_state': SEED, 'solver': 'liblinear', 'C': C_VALUE
        },
        'threshold_grid': [0.05, 0.95, 0.01],
        'threshold_selection_rule': 'f1_macro > f1_ia > recall_ia > distance_to_0.5',
    },
    'versions_des_bibliotheques': package_versions(['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'openpyxl', 'joblib', 'torch']),
    'threshold': float(threshold),
    'model_size_bytes': int(model_size_bytes_value),
    'training_time_seconds': float(training_time_seconds),
    'validation_inference_time_seconds': float(valid_inference_time),
    'test_inference_time_seconds': float(test_inference_time),
}
save_json(ARTIFACT_DIR / 'metadata.json', metadata)

print('Seuil retenu:', threshold)
display(pd.DataFrame([report_to_flat_row('tfidf_logistic_regression', validation_metrics), report_to_flat_row('tfidf_logistic_regression', test_metrics)]).assign(split=['validation', 'test']))


/tmp/ipykernel_64762/1486495466.py:56: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'date': pd.Timestamp.utcnow().isoformat(),


NameError: name 'shared_error_analysis' is not defined

In [ ]:

# Courbes, matrice de confusion, calibration et distribution des probabilités sur validation/test
print('Classification report validation')
print(classification_report(df_valid['is_ai'], (valid_scores >= threshold).astype(int), target_names=['non-IA', 'IA'], zero_division=0))
print('Classification report test')
print(classification_report(df_test['is_ai'], (test_scores >= threshold).astype(int), target_names=['non-IA', 'IA'], zero_division=0))

plot_confusion_matrix(validation_report, title='Matrice de confusion — validation')
plot_confusion_matrix(test_report, title='Matrice de confusion — test')

plot_roc_pr_calibration(validation_report, prefix='ml_validation', output_dir=ARTIFACT_DIR)
plot_roc_pr_calibration(test_report, prefix='ml_test', output_dir=ARTIFACT_DIR)

plot_probability_distribution(valid_scores, title='Distribution des probabilités — validation')
plot_probability_distribution(test_scores, title='Distribution des probabilités — test')
plot_threshold_metrics(threshold_table_validation, title='Scores selon le seuil — validation')

fig, ax = plt.subplots(figsize=(6, 4))
positive_features.head(20).iloc[::-1].plot(kind='barh', x='feature', y='weight', ax=ax, color='#f58518', legend=False)
ax.set_title('Top coefficients positifs pour IA')
ax.set_xlabel('Poids')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
negative_features.head(20).iloc[::-1].plot(kind='barh', x='feature', y='weight', ax=ax, color='#4c78a8', legend=False)
ax.set_title('Top coefficients négatifs pour non-IA')
ax.set_xlabel('Poids')
plt.tight_layout()
plt.show()

print('Erreurs validation:')
display(validation_report.classification_report)
print('Erreurs test:')
display(pd.DataFrame({
    'source_index': errors_test['source_index'],
    'record_id': errors_test['record_id'],
    'texte': errors_test['text'],
    'vrai': errors_test['true_label'],
    'prédit': errors_test['predicted_label'],
    'probabilité_ia': errors_test['probability_ia'],
    'type_erreur': errors_test['error_type'],
}))

manual_examples = [
    'Entraîner un réseau neuronal convolutif',
    'Construire un pipeline RAG',
    'Déployer un modèle de machine learning',
    'Gérer la relation client',
    'Préparer une réunion commerciale',
    'Utiliser Excel pour suivre un budget',
    '',
    'IA',
    'é',
    'zqxwplm nrvq tppp',
    'Cette compétence vise à automatiser la classification de documents et la génération de réponses avec un modèle de langage dans un contexte métier très concret.'
]
manual_results = []
for example in manual_examples:
    result = predict_competence_ml(example)
    manual_results.append({
        'modèle': 'ML',
        **result,
    })
manual_df = pd.DataFrame(manual_results)
display(manual_df)



Les artefacts et métriques sont enregistrés dans : `artifacts/classification_ia_non_ia_ml/`

Le split commun est conservé dans : `artifacts/classification_ia_non_ia_common/splits.csv`
